In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load UK processed dataset

df = pd.read_csv(
    "../data/processed/uk_processed.csv"
)

print(df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
severity_counts = df["severity_class"].value_counts()

print(severity_counts)

print("\nPercentage:")
print(
    (severity_counts / len(df) * 100).round(2)
)

In [ ]:
X = df.drop(
    columns=["severity_class"]
)

y = df["severity_class"]


print(X.shape)
print(y.shape)

In [ ]:
drop_columns = [
    "collision_index",
    "collision_ref_no",
    "date"
]


X = X.drop(
    columns=drop_columns,
    errors="ignore"
)


print(X.shape)

In [ ]:
df.shape

In [ ]:
df["severity_class"].value_counts()

In [ ]:
X.shape

In [ ]:
# بررسی ستون های ورودی مدل

print("Number of features:", X.shape[1])

print("\nColumns:")
for col in X.columns:
    print(col)

In [ ]:
# جدا کردن ستون های عددی و کیفی

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
for col in categorical_features:
    print("\n---", col, "---")
    print(
        X[col].nunique()
    )

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(
    label_encoder.classes_
)

In [ ]:
severity_mapping = pd.DataFrame({
    "Original": label_encoder.classes_,
    "Encoded": label_encoder.transform(
        label_encoder.classes_
    )
})

severity_mapping

In [ ]:
print(numeric_features)

In [ ]:
print(categorical_features)

In [ ]:
drop_categorical = [
    "time",
    "local_authority_ons_district",
    "local_authority_highway",
    "local_authority_highway_current",
    "lsoa_of_accident_location"
]


X = X.drop(
    columns=drop_categorical,
    errors="ignore"
)


print(X.shape)

print(
    X.select_dtypes(include="object").columns.tolist()
)

In [ ]:
X.shape

In [ ]:
# بررسی نهایی ستون های کیفی قبل از Encoding

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))


print("\nNumeric features:")
print(numeric_features)

print("\nNumber of numeric features:", len(numeric_features))

In [ ]:
# حذف نهایی Leakage و Featureهای غیرقابل تعمیم

remove_features = [
    # Leakage
    "collision_severity",
    "enhanced_severity_collision",
    "collision_injury_based",
    "collision_adjusted_severity_serious",
    "collision_adjusted_severity_slight",

    # Geographic identifiers
    "location_easting_osgr",
    "location_northing_osgr",
    "longitude",
    "latitude",

    # Administrative identifiers
    "police_force",
    "local_authority_district",
    "local_authority_ons_district",
    "local_authority_highway",
    "local_authority_highway_current",
    "lsoa_of_accident_location",

    # Redundant time field
    "time"
]


X = X.drop(
    columns=remove_features,
    errors="ignore"
)


print("New shape:")
print(X.shape)


print("\nRemaining object columns:")
print(
    X.select_dtypes(include="object")
    .columns
    .tolist()
)

In [ ]:
X.shape

In [ ]:
# بررسی نهایی Featureهای باقی‌مانده قبل از Encoding

print("Columns:")
for i, col in enumerate(X.columns, 1):
    print(i, col)


print("\nObject columns:")
print(
    X.select_dtypes(include="object")
    .columns
    .tolist()
)

In [ ]:
redundant_features = [
    "collision_year",
    "day_of_week",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "junction_detail",
    "junction_detail_historic",
    "junction_control",
    "special_conditions_at_site",
    "carriageway_hazards_historic",
    "carriageway_hazards"
]


X = X.drop(
    columns=redundant_features,
    errors="ignore"
)


print(X.shape)

print(
    X.columns.tolist()
)

In [ ]:
# Final target separation

y = df["severity_class"]

print(X.shape)
print(y.shape)

print(y.value_counts())

In [ ]:
# Final feature types

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Numeric:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer


# Preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)


X_encoded = preprocessor.fit_transform(X)


print("Encoded shape:")
print(X_encoded.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)


print(label_encoder.classes_)

In [ ]:
X_encoded.shape

In [ ]:
label_encoder.classes_

In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


print("Train:")
print(X_train.shape)

print("\nTest:")
print(X_test.shape)

In [ ]:
print("Train distribution:")
print(pd.Series(y_train).value_counts(normalize=True))


print("\nTest distribution:")
print(pd.Series(y_test).value_counts(normalize=True))

## 1. Random Forest Classifier

Random Forest is used as the first baseline machine learning model for accident severity classification.

This model was selected because it:
- Handles nonlinear relationships between accident characteristics and severity levels.
- Performs well on structured/tabular datasets.
- Provides feature importance measures for model interpretation.
- Can handle complex interactions among environmental, road, and vehicle-related factors.

### Model Configuration

The model is trained with:
- 300 decision trees (`n_estimators=300`)
- Balanced class weights to address the severe class imbalance, especially the low frequency of fatal accidents.
- Fixed random state (`random_state=42`) to ensure reproducibility.

The model performance is evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- Macro F1-score
- Confusion Matrix

In [ ]:
from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)


rf_model.fit(
    X_train,
    y_train
)


print("Random Forest trained successfully")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


y_pred_rf = rf_model.predict(X_test)


print("Accuracy:",
      round(accuracy_score(y_test, y_pred_rf),4))


print("Macro F1:",
      round(
          f1_score(
              y_test,
              y_pred_rf,
              average="macro"
          ),
          4
      ))


print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=label_encoder.classes_
    )
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


cm = confusion_matrix(
    y_test,
    y_pred_rf
)


disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)


disp.plot()

plt.title("Random Forest Confusion Matrix")

plt.show()

## 2. XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is applied as an advanced ensemble learning model for accident severity classification.

Unlike bagging-based methods such as Random Forest, XGBoost builds decision trees sequentially, where each new tree attempts to correct the errors of previous trees. This allows the model to capture complex nonlinear relationships between road, environmental, and vehicle-related factors.

XGBoost was selected because it:
- Provides strong predictive performance on structured/tabular datasets.
- Can model complex interactions among accident characteristics.
- Includes regularization mechanisms to reduce overfitting.
- Supports weighted training to address severe class imbalance in accident severity classes.

### Class Imbalance Handling

The UK accident dataset contains a highly imbalanced severity distribution, where fatal accidents represent a small minority compared with slight and serious accidents.

To reduce the bias toward majority classes, sample weights are calculated using balanced weighting and incorporated during model training.

### Model Evaluation

The XGBoost model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Macro F1-score
- Confusion Matrix

Special attention is given to minority-class performance, particularly Fatal and Serious accident detection.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print(sample_weights[:10])

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)


xgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)


print("XGBoost trained successfully")

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)


print("Accuracy:",
      round(accuracy_score(y_test, y_pred_xgb),4))


print("\nMacro F1:",
      round(
          f1_score(
              y_test,
              y_pred_xgb,
              average="macro"
          ),
          4
      ))


print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=label_encoder.classes_
    )
)

## 3. CatBoost Classifier

CatBoost is applied as an advanced gradient boosting algorithm designed for tabular datasets with categorical variables.

Unlike traditional encoding approaches, CatBoost can directly process categorical features using ordered target statistics, reducing information loss caused by one-hot encoding.

CatBoost was selected because it:
- Provides strong performance on heterogeneous tabular data.
- Handles categorical variables efficiently.
- Includes regularization mechanisms to reduce overfitting.
- Provides compatibility with SHAP-based explainability.

The model is evaluated using the same metrics used for previous models:
- Accuracy
- Precision
- Recall
- F1-score
- Macro F1-score
- Confusion Matrix

In [ ]:
# Prepare data for CatBoost

X_cat = X.copy()

y_cat = y_encoded.copy()


print(X_cat.shape)
print(y_cat.shape)

In [ ]:
from sklearn.model_selection import train_test_split


X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)


print(X_train_cat.shape)
print(X_test_cat.shape)

In [ ]:
cat_features_indices = [
    X_train_cat.columns.get_loc(col)
    for col in categorical_features
]


print(cat_features_indices)

In [ ]:
X_train_cat.shape

In [ ]:
X_test_cat.shape

In [ ]:
cat_features_indices

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100,
    auto_class_weights="Balanced"
)


cat_model.fit(
    X_train_cat,
    y_train_cat,
    cat_features=cat_features_indices
)


print("CatBoost trained successfully")

In [ ]:
y_pred_cat = cat_model.predict(
    X_test_cat
)


# تبدیل شکل خروجی CatBoost
y_pred_cat = y_pred_cat.astype(int).ravel()


print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test_cat,
            y_pred_cat
        ),
        4
    )
)


print(
    "\nMacro F1:",
    round(
        f1_score(
            y_test_cat,
            y_pred_cat,
            average="macro"
        ),
        4
    )
)


print("\nClassification Report:\n")

print(
    classification_report(
        y_test_cat,
        y_pred_cat,
        target_names=label_encoder.classes_
    )
)

## Model Performance Comparison

Three ensemble learning models were evaluated for multi-class accident severity classification: Random Forest, XGBoost, and CatBoost.

Due to severe class imbalance, accuracy alone was not considered sufficient. Therefore, macro F1-score and class-specific recall values, especially for Fatal and Serious classes, were emphasized.


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "CatBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test_cat, y_pred_cat)
    ],
    "Macro_F1": [
        f1_score(y_test, y_pred_rf, average="macro"),
        f1_score(y_test, y_pred_xgb, average="macro"),
        f1_score(y_test_cat, y_pred_cat, average="macro")
    ]
})


results

In [ ]:
results.to_csv(
    "../outputs/tables/model_comparison_baseline.csv",
    index=False
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


model_results = pd.DataFrame({

    "Model": [
        "Random Forest",
        "XGBoost",
        "CatBoost"
    ],

    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test_cat, y_pred_cat)
    ],

    "Precision_macro": [
        precision_score(y_test, y_pred_rf, average="macro"),
        precision_score(y_test, y_pred_xgb, average="macro"),
        precision_score(y_test_cat, y_pred_cat, average="macro")
    ],

    "Recall_macro": [
        recall_score(y_test, y_pred_rf, average="macro"),
        recall_score(y_test, y_pred_xgb, average="macro"),
        recall_score(y_test_cat, y_pred_cat, average="macro")
    ],

    "F1_macro": [
        f1_score(y_test, y_pred_rf, average="macro"),
        f1_score(y_test, y_pred_xgb, average="macro"),
        f1_score(y_test_cat, y_pred_cat, average="macro")
    ],

    "Fatal_Recall": [
        recall_score(y_test, y_pred_rf, labels=[0], average=None)[0],
        recall_score(y_test, y_pred_xgb, labels=[0], average=None)[0],
        recall_score(y_test_cat, y_pred_cat, labels=[0], average=None)[0]
    ],

   "Serious_Recall": [
    recall_score(
        y_test,
        y_pred_rf,
        labels=[1],
        average=None
    )[0],

    recall_score(
        y_test,
        y_pred_xgb,
        labels=[1],
        average=None
    )[0],

    recall_score(
        y_test_cat,
        y_pred_cat,
        labels=[1],
        average=None
    )[0]
]})


model_results.round(4)

In [ ]:
# Save baseline model comparison results

model_results.to_csv(
    "../outputs/tables/model_comparison_baseline.csv",
    index=False
)

print("Model comparison table saved successfully.")

## Baseline Model Comparison

Three ensemble learning models were evaluated for multi-class accident severity classification.

Because of the severe class imbalance among severity categories, model evaluation was performed using multiple metrics, including accuracy, macro F1-score, and class-specific recall.

Macro F1-score was considered to evaluate balanced performance across all severity classes, while Fatal and Serious recall values were specifically analyzed because these classes represent high-impact accident outcomes.